In [72]:
import os
import pandas as pd
import numpy as np
import matplotlib as plt
import networkx as nx
import mip

BUILDING_NAME = "Edificio2.csv"

ASCENDING_SPEED = 1.0
DESCENDING_SPEED = 2.0
HORIZONTAL_SPEED = 1.5

ASCENDING_ENERGY = 50
DESCENDING_ENERGY = 5
HORIZONTAL_ENERGY = 10

X_BASE_RANGE = range(-8, 6) if "Edificio1.csv" == BUILDING_NAME else range(-10, 11)
Y_BASE_RANGE = range(-17, -14) if "Edificio1.csv" == BUILDING_NAME else range(-31, -29)
Z_BASE = 0

BATTERY_CAPACITY = 3600 if "Edificio1.csv" == BUILDING_NAME else 21600  # 1 Wh is 3600 J

Y_ENTRY_POINT = -12.5 if "Edificio1.csv" == BUILDING_NAME else -20.0

# Reading CSV and creating pairs


In [73]:
# loading building csv data

nodes = pd.read_csv(BUILDING_NAME)

# Creating Bases and entry points

In [74]:
building.index += 1
# Adding is_base and is_entry_point columns


nodes["is_base"] = False
nodes["is_entry_point"] = nodes["y"] <= Y_ENTRY_POINT

# Genero tutte le basi candidate come prima
bases = []
for x in X_BASE_RANGE:
    for y in Y_BASE_RANGE:
        bases.append(
            {"x": x, "y": y, "z": Z_BASE, "is_base": True, "is_entry_point": False}
        )

bases_df = pd.DataFrame(bases)

# Calcolo del centroide sui nodi di servizio
centroid_x = nodes["x"].mean()
centroid_y = nodes["y"].mean()

# Seleziono la base candidata più vicina al centroide (distanza euclidea su x,y)
bases_df["dist_centroid"] = np.hypot(
    bases_df["x"] - centroid_x, bases_df["y"] - centroid_y
)
nearest_idx = bases_df["dist_centroid"].idxmin()
selected_base = bases_df.loc[[nearest_idx]].drop(columns=["dist_centroid"])

# Mantengo solo i nodi di servizio + la base più vicina al centroide
building = pd.concat([nodes, selected_base], ignore_index=True)
active_base = building.index[-1]  # Definisco l'indice della base selezionata

building.to_csv("building.csv", index=False)
print(
    f"Base scelta (indice {active_base}): "
    f"(x={selected_base.iloc[0]['x']}, y={selected_base.iloc[0]['y']}, z={Z_BASE})"
)

Base scelta (indice 696): (x=-3, y=-30, z=0)


# Merging nodes to create pairs

In [75]:
# generating all possible pairs of points (edges)
nodes_pairs = (
    building.rename_axis("source_node")
    .reset_index()
    .merge(building.rename_axis("target_node").reset_index(), how="cross")
)

# removing self-self pairs
nodes_pairs = nodes_pairs[
    nodes_pairs["source_node"] != nodes_pairs["target_node"]
]  # keeping only pairs with different points

In [76]:
first_point_cols = ["x_x", "y_x", "z_x"]
second_point_cols = ["x_y", "y_y", "z_y"]
diff_cols = ["dx", "dy", "dz"]

# Computing distances 

In [77]:
# function to determine if two points are connected based on distance and coordinate deltas


def process_connections(dataframe):
    # distances and deltas on coordinates
    dataframe[diff_cols] = (
        dataframe[second_point_cols].values - dataframe[first_point_cols].values
    )
    dataframe["euclidean_distance"] = np.sqrt(
        (dataframe[diff_cols] ** 2).sum(axis=1)
    )  # axis=1 sum on columns

    # conditions near and tunnel
    condition_near = dataframe["euclidean_distance"] <= 4.0

    condition_tunnel = (
        ((dataframe["dx"].abs() <= 0.5) & (dataframe["dy"].abs() <= 0.5))
        | ((dataframe["dy"].abs() <= 0.5) & (dataframe["dz"].abs() <= 0.5))
        | ((dataframe["dx"].abs() <= 0.5) & (dataframe["dz"].abs() <= 0.5))
    )

    condition_tunnel_and_not_far = (
        dataframe["euclidean_distance"] <= 11.0
    ) & condition_tunnel

    # if the arc is from base to entry point or viceversa
    condition_base_to_entry_point = (
        (dataframe["is_base_x"]) & (dataframe["is_entry_point_y"])
    ) | ((dataframe["is_base_y"]) & (dataframe["is_entry_point_x"]))

    # l'unica connessione ammessa è base <-> entry point
    condition_involves_base = dataframe["is_base_x"] | dataframe["is_base_y"]

    # Gli archi tra nodi griglia usano le regole di distanza (near/tunnel)
    # Gli archi con basi usano SOLO la regola base <-> entry point
    dataframe["are_connected"] = (
        (condition_near & ~condition_involves_base)
        | (condition_tunnel_and_not_far & ~condition_involves_base)
        | condition_base_to_entry_point
    )
    return dataframe

In [78]:
nodes_pairs = process_connections(nodes_pairs)

nodes_pairs.to_csv("pairs.csv", index=False)

nodes_pairs.head()

,source_node,x_x,y_x,z_x,is_base_x,is_entry_point_x,target_node,x_y,y_y,z_y,is_base_y,is_entry_point_y,dx,dy,dz,euclidean_distance,are_connected
1,0,-15.7277,-18.288,79.3699,False,False,1,-16.9469,-14.8742,79.3699,False,False,-1.2192,3.4138,0.0,3.624980,True
2,0,-15.7277,-18.288,79.3699,False,False,2,-16.9469,-7.2542,79.3699,False,False,-1.2192,11.0338,0.0,11.100955,False
3,0,-15.7277,-18.288,79.3699,False,False,3,-19.1110,-7.2542,79.3699,False,False,-3.3833,11.0338,0.0,11.540861,False
4,0,-15.7277,-18.288,79.3699,False,False,4,-20.7264,-5.9131,79.3699,False,False,-4.9987,12.3749,0.0,13.346353,False
5,0,-15.7277,-18.288,79.3699,False,False,5,-20.7264,-3.4138,79.3699,False,False,-4.9987,14.8742,0.0,15.691680,False


In [79]:
# computing vertical distances on z and distances on horizontal plane x,y
nodes_pairs["d_horizontal"] = np.sqrt(nodes_pairs["dx"] ** 2 + nodes_pairs["dy"] ** 2)
nodes_pairs["d_vertical"] = nodes_pairs["dz"]

# dropping columns that are not useful anymore
nodes_pairs.drop(columns=["dx", "dy", "dz"], inplace=True)
nodes_pairs.drop(columns=["x_x", "y_x", "z_x", "x_y", "y_y", "z_y"], inplace=True)

# keeping only connected pairs
nodes_pairs = nodes_pairs[nodes_pairs["are_connected"] == True]
nodes_pairs.drop(columns=["are_connected"], inplace=True)

nodes_pairs.head(5)

,source_node,is_base_x,is_entry_point_x,target_node,is_base_y,is_entry_point_y,euclidean_distance,d_horizontal,d_vertical
1,0,False,False,1,False,False,3.62498,3.62498,0.0000
40,0,False,False,40,False,False,3.84050,3.84050,0.0000
41,0,False,False,41,False,False,4.38910,0.00000,-4.3891
82,0,False,False,82,False,False,9.87550,0.00000,-9.8755
653,0,False,False,653,False,False,7.74190,7.74190,0.0000


# Function for time travel

In [80]:
# finding rows corresponding to different movement types

d_horizontal = nodes_pairs["d_horizontal"]
d_vertical = nodes_pairs["d_vertical"]
rows_horizontal = d_vertical == 0
rows_ascending = (d_horizontal == 0) & (d_vertical > 0)
rows_descending = (d_horizontal == 0) & (d_vertical < 0)
rows_oblique_ascending = (d_horizontal != 0) & (d_vertical > 0)
rows_oblique_descending = (d_horizontal != 0) & (d_vertical < 0)

In [81]:
# function for time travel based on movement types and speeds


def compute_time_travel(dataframe):
    dataframe.loc[rows_horizontal, "time_travel"] = (
        dataframe.loc[rows_horizontal, "d_horizontal"] / HORIZONTAL_SPEED
    )

    dataframe.loc[rows_ascending, "time_travel"] = (
        dataframe.loc[rows_ascending, "d_vertical"] / ASCENDING_SPEED
    )
    dataframe.loc[rows_descending, "time_travel"] = (
        -dataframe.loc[rows_descending, "d_vertical"] / DESCENDING_SPEED
    )

    dataframe.loc[rows_oblique_ascending, "time_travel"] = np.maximum(
        dataframe.loc[rows_oblique_ascending, "d_horizontal"] / HORIZONTAL_SPEED,
        dataframe.loc[rows_oblique_ascending, "d_vertical"] / ASCENDING_SPEED,
    )

    dataframe.loc[rows_oblique_descending, "time_travel"] = np.maximum(
        dataframe.loc[rows_oblique_descending, "d_horizontal"] / HORIZONTAL_SPEED,
        -dataframe.loc[rows_oblique_descending, "d_vertical"] / DESCENDING_SPEED,
    )
    return dataframe

# Function for energy consumption


In [82]:
# function for energy cost based on movement types and energy consumption rates


def compute_energy_cost(dataframe):
    dataframe.loc[rows_horizontal, "energy_consumption"] = (
        dataframe.loc[rows_horizontal, "d_horizontal"] * HORIZONTAL_ENERGY
    )

    dataframe.loc[rows_ascending, "energy_consumption"] = (
        dataframe.loc[rows_ascending, "d_vertical"] * ASCENDING_ENERGY
    )
    dataframe.loc[rows_descending, "energy_consumption"] = (
        -dataframe.loc[rows_descending, "d_vertical"] * DESCENDING_ENERGY
    )

    dataframe.loc[rows_oblique_ascending, "energy_consumption"] = (
        dataframe.loc[rows_oblique_ascending, "d_horizontal"] * HORIZONTAL_ENERGY
        + dataframe.loc[rows_oblique_ascending, "d_vertical"] * ASCENDING_ENERGY
    )

    dataframe.loc[rows_oblique_descending, "energy_consumption"] = (
        dataframe.loc[rows_oblique_descending, "d_horizontal"] * HORIZONTAL_ENERGY
        + -dataframe.loc[rows_oblique_descending, "d_vertical"] * DESCENDING_ENERGY
    )
    return dataframe

# Actually computing energy and time costs for arcs

In [83]:
# computing time travel and energy cost for each pair (edges)

nodes_pairs = compute_time_travel(nodes_pairs)

nodes_pairs = compute_energy_cost(nodes_pairs)

# rounding to 2 decimal places
nodes_pairs["time_travel"] = nodes_pairs["time_travel"].round(2)
nodes_pairs["energy_consumption"] = nodes_pairs["energy_consumption"].round(2)

nodes_pairs.drop(
    columns=["d_horizontal", "d_vertical", "euclidean_distance"], inplace=True
)

nodes_pairs.head(10)

,source_node,is_base_x,is_entry_point_x,target_node,is_base_y,is_entry_point_y,time_travel,energy_consumption
1,0,False,False,1,False,False,2.42,36.25
40,0,False,False,40,False,False,2.56,38.40
41,0,False,False,41,False,False,2.19,21.95
82,0,False,False,82,False,False,4.94,49.38
653,0,False,False,653,False,False,5.16,77.42
697,1,False,False,0,False,False,2.42,36.25
699,1,False,False,2,False,False,5.08,76.20
739,1,False,False,42,False,False,2.19,21.95
780,1,False,False,83,False,False,4.94,49.38
1395,2,False,False,1,False,False,5.08,76.20


In [84]:
# Pruning degli archi: manteniamo gli archi più corti sia in uscita che in entrata per garantire la connettività
k_keep = 80

# Archi migliori in uscita (per garantire out-degree >= 1)
out_edges = nodes_pairs.sort_values("time_travel").groupby("source_node").head(k_keep)

# Archi migliori in entrata (per garantire in-degree >= 1)
in_edges = nodes_pairs.sort_values("time_travel").groupby("target_node").head(k_keep)

# Unione degli archi per evitare nodi isolati
nodes_pairs = pd.concat([out_edges, in_edges]).drop_duplicates()

print(f"Pruning completato. Archi rimanenti: {len(nodes_pairs)}")

Pruning completato. Archi rimanenti: 4672


# Creating Graph with Networkx


In [85]:
# creating directed graph G from the dataframe of edges with costs

G = nx.from_pandas_edgelist(
    nodes_pairs,  # dataframe containing edges
    source="source_node",  # indicating the start of the edge
    target="target_node",  # indicating the end of the edge
    edge_attr=["time_travel", "energy_consumption"],  # edge costs
    create_using=nx.DiGraph(),  # creating an oriented graph
)

# nx.draw_networkx(G, node_size=50, font_size=8)

# Clustering per viaggi (zone connesse base↔nodi)

In [86]:
from scipy.cluster.vq import kmeans2

K_WANTED = 8
MAX_CLUSTER_SIZE = 120
TARGET_MIN = 70  # Soglia minima per evitare viaggi troppo brevi

# 1. Filtro nodi validi (Raggiungibili globalmente dalla base)
service_nodes = building[building["is_base"] == False].index.tolist()
reachable_from_base = nx.single_source_dijkstra_path_length(
    G, active_base, weight="energy_consumption"
)
reachable_to_base = nx.single_source_dijkstra_path_length(
    G.reverse(), active_base, weight="energy_consumption"
)
valid_nodes = [
    n for n in service_nodes if n in reachable_from_base and n in reachable_to_base
]

# 2. K-means Puro: Ogni nodo appartiene a un solo gruppo geografico
coords_array = building.loc[valid_nodes, ["x", "y", "z"]].to_numpy()
centroidi, labels = kmeans2(coords_array, K_WANTED, minit="points", iter=30)

trips = [[] for _ in range(K_WANTED)]
for idx, label in enumerate(labels):
    trips[label].append(valid_nodes[idx])

# 3. Bilanciamento dimensioni (Split dei troppo grandi)
final_trips = []
for t in trips:
    if len(t) > MAX_CLUSTER_SIZE:
        final_trips.append(t[: len(t) // 2])
        final_trips.append(t[len(t) // 2 :])
    elif t:
        final_trips.append(t)

trips = final_trips
print(f"Cluster geografici creati: {len(trips)}")
print("Nodi per cluster:", [len(t) for t in trips])
TRIPS_READY = True

Cluster geografici creati: 8
Nodi per cluster: [77, 81, 95, 79, 100, 94, 93, 77]


In [87]:
def diagnose_cluster(cluster_idx, t_nodes, base_node, G_full, capacity):
    """
    Diagnostica dettagliata di un cluster:
    - Verifica connettività dalla base
    - Verifica raggiungibilità dei nodi tra loro
    - Stima energetica realistica per un tour TSP sommando i costi degli archi
    """

    diag = {
        "cluster_idx": cluster_idx,
        "num_nodes": len(t_nodes),
        "reachable_from_base": [],
        "can_return_to_base": [],
        "unreachable_nodes": [],
        "issues": [],
        "warnings": [],
        "total_min_energy_required": 0,
        "max_single_ar": 0,
    }

    # Calcola distanze minime dalla base a tutti i nodi (via Dijkstra)
    dist_from_base = nx.single_source_dijkstra_path_length(
        G_full, base_node, weight="energy_consumption"
    )

    # Calcola distanze minime dai nodi alla base
    dist_to_base = {}
    for node in t_nodes:
        try:
            path_length = nx.single_source_dijkstra_path_length(
                G_full, node, weight="energy_consumption"
            )
            if base_node in path_length:
                dist_to_base[node] = path_length[base_node]
        except:
            pass

    min_incoming_cost = (
        {}
    )  # Costo minimo di ingresso per ogni nodo (da altri nodi del cluster)
    min_outgoing_cost = (
        {}
    )  # Costo minimo di uscita per ogni nodo (verso altri nodi del cluster)
    max_single_ar = 0  # Il massimo A/R singolo per qualsiasi nodo

    # Costruisci set di nodi cluster per ricerche veloci
    N_cluster_set = set([base_node] + t_nodes)

    for node in t_nodes:
        # Controlla raggiungibilità dalla base
        if node in dist_from_base:
            diag["reachable_from_base"].append(node)
            energy_from_base = dist_from_base[node]
        else:
            diag["unreachable_nodes"].append(node)
            diag["issues"].append(f"Nodo {node}: NON raggiungibile dalla base")
            continue

        # Controlla se può tornare alla base
        if node in dist_to_base:
            diag["can_return_to_base"].append(node)
            energy_to_base = dist_to_base[node]
        else:
            diag["issues"].append(f"Nodo {node}: NON può tornare alla base")
            continue

        # Verifica vincolo energetico minimo (A/R singolo)
        min_round_trip = energy_from_base + energy_to_base
        max_single_ar = max(max_single_ar, min_round_trip)

        if min_round_trip > capacity:
            diag["warnings"].append(
                f"Nodo {node}: Viaggio A/R singolo ({min_round_trip:.1f}J) > Batteria ({capacity}J)"
            )

    # STIMA DELL'ENERGIA PER UN UNICO TOUR TSP:
    # Per ogni nodo nel cluster, trova il costo minimo degli archi di ingresso/uscita
    # dai/verso altri nodi nel cluster (non dalla base!)
    for node in t_nodes:
        # Costi di ingresso: archi (i, node) dove i è nel cluster
        incoming_costs = [
            G_full[i][node]["energy_consumption"]
            for i in N_cluster_set
            if i != node and G_full.has_edge(i, node)
        ]
        min_incoming_cost[node] = (
            min(incoming_costs) if incoming_costs else float("inf")
        )

        # Costi di uscita: archi (node, j) dove j è nel cluster
        outgoing_costs = [
            G_full[node][j]["energy_consumption"]
            for j in N_cluster_set
            if j != node and G_full.has_edge(node, j)
        ]
        min_outgoing_cost[node] = (
            min(outgoing_costs) if outgoing_costs else float("inf")
        )

    # Stima tour TSP: per un tour che visita tutti i nodi una volta,
    # approssimiamo come la somma dei costi minimi di ingresso per ogni nodo
    # (questo rappresenta il costo di "raggiungerlo" e poi continuare il tour)
    # PIÙ il costo di ritorno alla base dall'ultimo nodo
    valid_incoming = [c for c in min_incoming_cost.values() if c != float("inf")]
    valid_outgoing = [c for c in min_outgoing_cost.values() if c != float("inf")]

    if valid_incoming and valid_outgoing:
        # Stima pessimistica: somma dei costi minimi di ingresso + uscita
        # Questo rappresenta il "ciclo" per visitare ogni nodo
        estimate_tour = sum(valid_incoming) + max(valid_outgoing)
    else:
        estimate_tour = 0

    diag["total_min_energy_required"] = estimate_tour
    diag["max_single_ar"] = max_single_ar

    # Verifica connettività interna tra nodi del cluster
    # Controlla se tutti i nodi raggiungibili sono interconnessi
    for start_node in t_nodes:
        try:
            local_reach = nx.single_source_dijkstra_path_length(
                G_full, start_node, weight="energy_consumption"
            )
            for target_node in t_nodes:
                if target_node != start_node and target_node not in local_reach:
                    diag["issues"].append(
                        f"Connettività: Nodo {start_node} non raggiunge Nodo {target_node}"
                    )
                    break
        except:
            pass

    return diag


# Esegui diagnostica per ogni cluster
cluster_diagnostics = []
feasible_clusters = []
infeasible_clusters = []

for idx, t_nodes in enumerate(trips, start=1):
    diag = diagnose_cluster(idx, t_nodes, active_base, G, BATTERY_CAPACITY)
    cluster_diagnostics.append(diag)

    # Stampa risultati
    print(f"Cluster {idx}: {len(t_nodes)} nodi")
    print(
        f"  Base -> Nodi raggiungibili: {len(diag['reachable_from_base'])}/{len(t_nodes)}"
    )
    print(
        f"  Nodi -> Base raggiungibili: {len(diag['can_return_to_base'])}/{len(t_nodes)}"
    )

    if diag["unreachable_nodes"]:
        print(f"  ❌ Nodi non raggiungibili: {diag['unreachable_nodes']}")

    if diag["issues"]:
        print(f"  ⚠️  PROBLEMI RILEVATI:")
        for issue in diag["issues"]:
            print(f"     - {issue}")
        infeasible_clusters.append(idx)
    else:
        print(f"  ✅ Cluster connesso e raggiungibile dalla base")
        feasible_clusters.append(idx)

    if diag["warnings"]:
        print(f"  ⚠️  AVVERTIMENTI:")
        for warning in diag["warnings"]:
            print(f"     - {warning}")

    print(f"  📊 Stima ENERGIA PER TOUR TSP (sommando i costi degli archi):")
    print(f"     - Max A/R singolo nodo: {diag.get('max_single_ar', 0):.1f}J")
    print(f"     - Stima tour totale: {diag['total_min_energy_required']:.1f}J")
    print(f"     - Capacità batteria: {BATTERY_CAPACITY}J")
    if diag["total_min_energy_required"] > BATTERY_CAPACITY:
        ratio = diag["total_min_energy_required"] / BATTERY_CAPACITY
        print(f"     ⚠️  TOUR NON FATTIBILE IN UNA CARICA ({ratio:.1f}x capacità)")
    else:
        print(f"     ✅ Tour possibile in una carica")
    print()

print(f"\n{'='*70}")
print(f"RIEPILOGO DIAGNOSTICA")
print(f"{'='*70}")
print(f"✅ Cluster FEASIBLE (connessi e raggiungibili): {feasible_clusters}")
print(f"❌ Cluster INFEASIBLE (problemi di connettività): {infeasible_clusters}")
print(f"{'='*70}\n")

Cluster 1: 77 nodi
  Base -> Nodi raggiungibili: 77/77
  Nodi -> Base raggiungibili: 77/77
  ✅ Cluster connesso e raggiungibile dalla base
  📊 Stima ENERGIA PER TOUR TSP (sommando i costi degli archi):
     - Max A/R singolo nodo: 5719.3J
     - Stima tour totale: 2184.8J
     - Capacità batteria: 21600J
     ✅ Tour possibile in una carica

Cluster 2: 81 nodi
  Base -> Nodi raggiungibili: 81/81
  Nodi -> Base raggiungibili: 81/81
  ✅ Cluster connesso e raggiungibile dalla base
  📊 Stima ENERGIA PER TOUR TSP (sommando i costi degli archi):
     - Max A/R singolo nodo: 2322.1J
     - Stima tour totale: 2148.2J
     - Capacità batteria: 21600J
     ✅ Tour possibile in una carica

Cluster 3: 95 nodi
  Base -> Nodi raggiungibili: 95/95
  Nodi -> Base raggiungibili: 95/95
  ✅ Cluster connesso e raggiungibile dalla base
  📊 Stima ENERGIA PER TOUR TSP (sommando i costi degli archi):
     - Max A/R singolo nodo: 3282.1J
     - Stima tour totale: 2350.5J
     - Capacità batteria: 21600J
     ✅ T

In [88]:
# Diagnostica Avanzata: Verifica Fattibilità Globale ed Energetica
# Il drone deve poter fare Base -> Nodo -> Base restando nei limiti di batteria (C).


def check_trip_feasibility(t_nodes, base_node, G_full, capacity):
    """
    Verifica se un cluster è risolvibile:
    1. Ogni nodo ha un percorso bidirezionale (Base <-> Nodo).
    2. Il costo energetico minimo A/R (Base -> Nodo -> Base) è <= C.
    3. Esiste connettività interna nel grafo globale tra tutti i nodi del cluster.
    """
    # 1. Distanze dalla base (Shortest Path globale)
    try:
        dist_from = nx.single_source_dijkstra_path_length(
            G_full, base_node, weight="energy_consumption"
        )
        dist_to = nx.single_source_dijkstra_path_length(
            G_full.reverse(), base_node, weight="energy_consumption"
        )
    except Exception as e:
        return False, f"Errore Dijkstra: {str(e)}"

    for n in t_nodes:
        # Verifica esistenza fisica del percorso
        if n not in dist_from:
            return False, f"Nodo {n}: NON raggiungibile dalla base!"
        if n not in dist_to:
            return False, f"Nodo {n}: NON può tornare alla base!"

        # Verifica energetica minima (Base -> Nodo -> Base)
        # Se anche solo il viaggio verso UN nodo e ritorno eccede C, il cluster è infattibile.
        min_round_trip = dist_from[n] + dist_to[n]
        if min_round_trip > capacity:
            return (
                False,
                f"Nodo {n}: A/R energetico ({min_round_trip:.1f}) > Batteria ({capacity})",
            )

    # 2. Verifica connettività interna (Transitive Closure)
    # Garantisce che il drone possa muoversi tra i nodi assegnati al cluster.
    for start_node in t_nodes:
        local_reach = nx.single_source_dijkstra_path_length(
            G_full, start_node, weight="energy_consumption"
        )
        for target_node in t_nodes:
            if target_node not in local_reach:
                return (
                    False,
                    f"Connettività: {start_node} non raggiunge {target_node} globalmente.",
                )

    return True, "OK (Fattibile)"


# --- Esecuzione Verifiche ---

# Nodi servizio totali vs Nodi nel cluster
covered = set().union(*trips) if trips else set()
s_nodes_list = building[building["is_base"] == False].index.tolist()

missing_total = sorted(list(set(s_nodes_list) - covered))
if missing_total:
    print(
        f"ATTENZIONE: {len(missing_total)} nodi non sono stati assegnati ad alcun viaggio!"
    )
else:
    print("OK: Tutti i nodi servizio sono coperti dai cluster.")

print(f"\nVerifica rigorosa (A/R Base + Connettività + Batteria {BATTERY_CAPACITY}J):")
all_ok = True
for idx, t_nodes in enumerate(trips, start=1):
    feasible, msg = check_trip_feasibility(t_nodes, active_base, G, BATTERY_CAPACITY)
    print(f"Viaggio {idx}: {len(t_nodes)} nodi | {msg}")
    if not feasible:
        all_ok = False

if all_ok:
    print(
        "\nPromosso! Tutti i cluster garantiscono la possibilità di completare il viaggio."
    )

OK: Tutti i nodi servizio sono coperti dai cluster.

Verifica rigorosa (A/R Base + Connettività + Batteria 21600J):
Viaggio 1: 77 nodi | OK (Fattibile)
Viaggio 2: 81 nodi | OK (Fattibile)
Viaggio 3: 95 nodi | OK (Fattibile)
Viaggio 4: 79 nodi | OK (Fattibile)
Viaggio 5: 100 nodi | OK (Fattibile)
Viaggio 6: 94 nodi | OK (Fattibile)
Viaggio 7: 93 nodi | OK (Fattibile)
Viaggio 8: 77 nodi | OK (Fattibile)

Promosso! Tutti i cluster garantiscono la possibilità di completare il viaggio.


# MODELLO MIP

In [89]:
# LOOP SU TUTTI I CLUSTER
# Per ogni cluster risolviamo un MIP separato

results = []  # Risultati di ottimizzazione per ogni cluster

for cluster_idx, t_nodes in enumerate(trips, start=1):
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_idx}: {len(t_nodes)} nodi - {t_nodes}")
    print(f"{'='*70}")

    # ===============================================================
    # STEP 1: CREIAMO I SET PER QUESTO CLUSTER
    # ===============================================================
    N_cluster = [active_base] + t_nodes  # All nodes in cluster (base + service nodes)
    S_cluster = t_nodes  # Service nodes in cluster
    B_cluster = [active_base]  # Base node

    # Archi: solo quelli tra nodi del cluster
    A_cluster = [(u, v) for u, v in G.edges if u in N_cluster and v in N_cluster]

    print(f"Set N_cluster: {N_cluster}")
    print(f"Set S_cluster: {S_cluster}")
    print(f"Number of arcs |A|: {len(A_cluster)}")

    # ===============================================================
    # STEP 2: CREIAMO IL MODELLO MIP PER QUESTO CLUSTER
    # ===============================================================
    m_cluster = mip.Model(
        f"DroneTourPlanning_Cluster_{cluster_idx}", solver_name=mip.CBC
    )
    m_cluster.threads = -1
    m_cluster.max_mip_gap = 100.0  # Ferma appena trova una soluzione ammissibile

    # Parametri per questo cluster
    t_cluster = {
        (u, v): data["time_travel"]
        for u, v, data in G.edges(data=True)
        if (u, v) in A_cluster
    }
    e_cluster = {
        (u, v): data["energy_consumption"]
        for u, v, data in G.edges(data=True)
        if (u, v) in A_cluster
    }
    C_cluster = BATTERY_CAPACITY

    # Calcola H (numero di tour) per questo cluster
    import math

    min_energy_in_out_cluster = {}
    for i in S_cluster:
        incoming = [e_cluster[j, i] for j in N_cluster if (j, i) in A_cluster]
        outgoing = [e_cluster[i, j] for j in N_cluster if (i, j) in A_cluster]
        min_energy_in = min(incoming) if incoming else 0
        min_energy_out = min(outgoing) if outgoing else 0
        min_energy_in_out_cluster[i] = min_energy_in + min_energy_out

    sum_min_energies_cluster = sum(min_energy_in_out_cluster.values())
    H_length_cluster = math.ceil((sum_min_energies_cluster / C_cluster) * 1.5)
    H_cluster = list(range(1, max(2, H_length_cluster + 1)))

    # Big M per questo cluster
    positive_energy_cluster = [v for v in e_cluster.values() if v > 0]
    M_cluster = len(S_cluster)

    print(f"H_cluster: {H_cluster}")
    print(f"M_cluster: {M_cluster}")

    # ===============================================================
    # STEP 3: CREIAMO LE VARIABILI
    # ===============================================================
    # x[i,j] = binary variable indicating if arc (i,j) is used
    x_cluster = {
        (i, j): m_cluster.add_var(var_type=mip.BINARY, name=f"x_{i}_{j}")
        for i, j in A_cluster
    }

    # f[i,j] = flow on arc (i,j) (for subtour elimination)
    f_cluster = {
        (i, j): m_cluster.add_var(lb=0, ub=len(S_cluster), name=f"f_{i}_{j}")
        for i, j in A_cluster
    }

    # ===============================================================
    # STEP 4: FUNZIONE OBIETTIVO
    # ===============================================================
    m_cluster.objective = mip.minimize(
        mip.xsum(t_cluster[i, j] * x_cluster[i, j] for i, j in A_cluster)
    )

    # ===============================================================
    # STEP 5: VINCOLI
    # ===============================================================
    # (2) Each service node is visited at least once
    for i in S_cluster:
        m_cluster += (
            mip.xsum(x_cluster[j, i] for j in N_cluster if (j, i) in A_cluster) >= 1
        )

    # (3) Flow conservation for x (arc variables)
    for i in N_cluster:
        m_cluster += mip.xsum(
            x_cluster[j, i] for j in N_cluster if (j, i) in A_cluster
        ) == mip.xsum(x_cluster[i, j] for j in N_cluster if (i, j) in A_cluster)

    # (5) Energy constraint
    m_cluster += (
        mip.xsum(e_cluster[i, j] * x_cluster[i, j] for i, j in A_cluster) <= C_cluster
    )

    # (6) SCF Subtour Elimination
    # (6a) f_ij <= M * x_ij (flow only on active arcs)
    for i, j in A_cluster:
        m_cluster += f_cluster[i, j] <= M_cluster * x_cluster[i, j]

    # (6b) Flow conservation at service nodes
    for i in S_cluster:
        inflow = mip.xsum(f_cluster[j, i] for j in N_cluster if (j, i) in A_cluster)
        outflow = mip.xsum(f_cluster[i, j] for j in N_cluster if (i, j) in A_cluster)
        visits = mip.xsum(x_cluster[j, i] for j in N_cluster if (j, i) in A_cluster)
        m_cluster += inflow - outflow == visits

    # ===============================================================
    # STEP 6: RISOLVIAMO IL MIP
    # ===============================================================
    print(f"Variables: {m_cluster.num_cols}, Constraints: {m_cluster.num_rows}")

    status = m_cluster.optimize(max_seconds=7200)  # 2 hours

    print(f"\nStatus: {status}")
    if m_cluster.objective_value is not None:
        print(f"Objective value: {m_cluster.objective_value:.2f}")

    # Salva il risultato
    results.append(
        {
            "cluster": cluster_idx,
            "nodes": t_nodes,
            "status": status,
            "objective": m_cluster.objective_value,
            "model": m_cluster,
            "x_vars": x_cluster,
            "f_vars": f_cluster,
        }
    )

print(f"\n{'='*70}")
print(f"OTTIMIZZAZIONE COMPLETATA PER TUTTI I {len(trips)} CLUSTER")
print(f"{'='*70}")


CLUSTER 1: 77 nodi - [20, 21, 22, 23, 24, 25, 26, 27, 61, 62, 63, 64, 65, 66, 67, 68, 101, 102, 103, 104, 105, 106, 107, 108, 109, 142, 143, 144, 145, 146, 147, 148, 149, 150, 183, 184, 185, 186, 187, 188, 189, 190, 191, 224, 225, 226, 227, 228, 229, 230, 231, 232, 265, 266, 267, 268, 269, 270, 271, 272, 273, 305, 306, 307, 308, 309, 310, 311, 344, 345, 346, 347, 348, 349, 350, 640, 643]
Set N_cluster: [696, 20, 21, 22, 23, 24, 25, 26, 27, 61, 62, 63, 64, 65, 66, 67, 68, 101, 102, 103, 104, 105, 106, 107, 108, 109, 142, 143, 144, 145, 146, 147, 148, 149, 150, 183, 184, 185, 186, 187, 188, 189, 190, 191, 224, 225, 226, 227, 228, 229, 230, 231, 232, 265, 266, 267, 268, 269, 270, 271, 272, 273, 305, 306, 307, 308, 309, 310, 311, 344, 345, 346, 347, 348, 349, 350, 640, 643]
Set S_cluster: [20, 21, 22, 23, 24, 25, 26, 27, 61, 62, 63, 64, 65, 66, 67, 68, 101, 102, 103, 104, 105, 106, 107, 108, 109, 142, 143, 144, 145, 146, 147, 148, 149, 150, 183, 184, 185, 186, 187, 188, 189, 190, 191, 224

In [34]:
# ESTRAZIONE E STAMPA DEI PERCORSI PER OGNI CLUSTER
# Adattamento del vecchio codice DFS per i nuovi risultati per cluster

print(f"\n{'='*70}")
print("STAMPA DEI PERCORSI OTTIMIZZATI PER OGNI CLUSTER")
print(f"{'='*70}\n")

all_paths_per_cluster = []

for result in results:
    cluster_idx = result["cluster"]
    t_nodes = result["nodes"]
    m_cluster = result["model"]
    x_vars = result["x_vars"]

    print(f"Cluster {cluster_idx} ({len(t_nodes)} nodi):")

    # Controlla se il modello ha una soluzione
    if m_cluster.num_solutions == 0:
        print(f"  ⚠️  Nessuna soluzione trovata per il cluster {cluster_idx}")
        continue

    # Estrai gli archi attivi (quelli con x=1)
    # Adattamento del vecchio codice: per ogni arco (i,j) in x_vars
    # se x_vars[i, j].x == 1.0, lo aggiungo a next_node
    next_node = {}
    for i, j in x_vars.keys():
        if x_vars[i, j].x is not None and x_vars[i, j].x == 1.0:
            # se è il primo arco da i creo una lista
            if i not in next_node:
                next_node[i] = []
            # inserisco il nodo di arrivo j dato il nodo di partenza i
            next_node[i].append(j)

    # Usa lo stesso algoritmo DFS del vecchio codice
    # lo stack sono i nodi da esplorare, parto dalla base
    stack = [active_base]
    path = []

    # finche c'è qualcosa nello stack vado avanti
    while stack:
        v = stack[-1]  # prendo ultimo elemento
        if v in next_node and next_node[v]:  # se v ha ancora archi non usati
            u = next_node[
                v
            ].pop()  # prendo un nodo target (un arco) e lo rimuovo dalla lista
            stack.append(u)  # metto la destinazione nello stack
        else:
            # se non ha archi uscenti li ho già esplorati e la inserisco nello stack
            path.append(stack.pop())

    # il percorso è al contrario
    path = path[::-1]

    # Formatta il percorso come richiesto
    path_str = ["0"]  # base
    for node in path[1:]:
        if node == active_base:
            path_str.append("-0")
        else:
            path_str.append("-" + str(node))

    print(f"  Viaggio {cluster_idx}: {''.join(path_str)}")
    all_paths_per_cluster.append(path)

print(f"\n{'='*70}")
print(f"RIEPILOGO: {len(results)} cluster elaborati")
print(f"{'='*70}")


STAMPA DEI PERCORSI OTTIMIZZATI PER OGNI CLUSTER

Cluster 1 (91 nodi):
  ⚠️  Nessuna soluzione trovata per il cluster 1
Cluster 2 (97 nodi):
  ⚠️  Nessuna soluzione trovata per il cluster 2
Cluster 3 (54 nodi):
  Viaggio 3: 0-158-117-76-35-651-650-645-646-34-32-31-32-33-646-30-29-70-71-72-73-72-71-112-111-152-193-234-233-232-191-150-109-68-27-641-644-645-642-28-69-110-151-192-233-234-193-194-195-0-157-156-155-196-198-197-198-196-195-196-155-154-113-114-113-154-153-152-153-154-155-156-157-116-115-116-34-33-74-75-74-0
Cluster 4 (104 nodi):
  ⚠️  Nessuna soluzione trovata per il cluster 4
Cluster 5 (71 nodi):
  Viaggio 5: 0-324-325-286-245-244-245-286-246-247-246-286-325-324-321-324-323-284-285-284-283-282-283-322-321-322-323-362-361-360-361-362-363-402-403-364-325-364-363-362-401-400-399-400-401-0-318-238-239-280-279-278-277-278-279-280-239-237-236-235-236-237-238-318-319-318-317-395-394-397-396-397-358-356-355-354-353-314-275-276-275-314-392-393-392-391-352-351-273-312-351-352-313-274-

# VARIABILI

In [ ]:
if m.objective_value:
    print(f"Obiettivo: {m.objective_value:.2f}")
if m.objective_bound:
    print(f"Lower bound: {m.objective_bound:.2f}")
gap = abs(m.objective_value - m.objective_bound) / abs(m.objective_value)
print(f"GAP: {gap:.4f}")
print(m.status)

In [ ]:
# Salvataggio del modello in formato leggibile
m.write("model.lp")

# Salvataggio della soluzione
if m.num_solutions:
    with open("solution.txt", "w") as file:
        file.write(f"Objective: {m.objective_value}\n")
        for var in m.vars:
            file.write(f"{var.name} = {var.x}\n")

In [ ]:
if m.num_solutions:

    # base unica già determinata
    active_base = B[0]
    # definisco per salvare i vari percorsi (path1, path2...)
    all_paths = []

    # in base al path in cui sono identifico gli archi che sono stati usati
    for idx, h in enumerate(H):
        # dizionario per salvare il nodo prossimo j dato il nodo i
        next_node = {}
        for i, j in A:
            if x[i, j, h].x == 1.0:
                # se è il primo arco creo una lista
                if i not in next_node:
                    next_node[i] = []
                # inserisco il nodo di arrivo j dato il nodo di partenza i
                # li metto in una lista perché dato un nodo i questo può avere più trasnizione di output
                next_node[i].append(j)

        # next_node[i] è la lista di tutti i nodi raggiungibili da i

        # lo stack sono i nodi da esplorare, parto dalla base
        stack = [active_base]
        path = []

        # finche c'è qualcosa nello stack vado avanti
        while stack:
            v = stack[-1]  # prendo ultimo elemento
            if v in next_node and next_node[v]:  # se v ha ancora archi non usati

                u = next_node[
                    v
                ].pop()  # prendo un nodo target (un arco) e lo rimuovo dalla lista
                stack.append(u)  # metto la destinazione nello stack
            else:
                # se non ha archi uscenti li ho già esplorati e la inserisco nello stack
                path.append(stack.pop())

        # il percorso è al contrario
        path = path[::-1]

        path_str = ["0"]  # base
        for node in path[1:]:
            if node == active_base:
                path_str.append("-0")
            else:
                path_str.append("-" + str(node))

        print(f"Viaggio {h}: {''.join(path_str)}")
        all_paths.append(path)

In [ ]:
# Validazione finale: Verifica copertura completa dei nodi S
visited_nodes = set()

# Un nodo i è visitato se esiste almeno un arco entrante attivo in un qualsiasi viaggio h
for i in S:
    is_visited = False
    for h in H:
        # Check incoming edges to i
        if any(x[j, i, h].x > 0.5 for j in N if (j, i) in A):
            is_visited = True
            break

    if is_visited:
        visited_nodes.add(i)

missing_nodes = set(S) - visited_nodes

if not missing_nodes:
    print(f"SUCCESS: Tutti i {len(S)} nodi di servizio sono stati visitati.")
else:
    print(f"WARNING: {len(missing_nodes)} nodi su {len(S)} NON sono stati visitati!")
    print(f"Lista nodi mancanti: {sorted(list(missing_nodes))}")

In [ ]:
import matplotlib.pyplot as plt

# creo una lista delle basi da rimuovere

remove_bases = [b for b in B if b != active_base]
# print(basi_da_rimuovere)

# creo il grafo senza le basi
G_sub = G.copy()
G_sub.remove_nodes_from(remove_bases)

plt.figure(figsize=(15, 12))

# k controlla la distanza tra i nodi
pos = nx.spring_layout(G_sub, k=2, iterations=50, seed=42)

# colore rosso per la base
node_colors = ["red" if n == active_base else "skyblue" for n in G_sub.nodes()]

# 0 se è la base altrimenti i valore del nodo
labels = {n: (0 if n == active_base else n) for n in G_sub.nodes()}

nx.draw_networkx(
    G_sub,
    pos=pos,
    labels=labels,
    node_size=500,
    node_color=node_colors,
    font_size=10,
    font_weight="bold",
    edge_color="gray",
    arrows=True,
    arrowsize=15,
)

plt.title(f"Graph with base {active_base}", fontsize=14)
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# creo una lista delle basi da rimuovere

remove_bases = [b for b in B if b != active_base]
# print(basi_da_rimuovere)

# creo il grafo senza le basi
G_sub = G.copy()
G_sub.remove_nodes_from(remove_bases)

plt.figure(figsize=(18, 14))

# k controlla la distanza tra i nodi
pos = nx.spring_layout(G_sub, k=2.5, iterations=100, seed=42)


node_colors = ["red" if n == active_base else "skyblue" for n in G_sub.nodes()]

# 0 se è la base altrimenti i valore del nodo
labels = {n: (0 if n == active_base else n) for n in G_sub.nodes()}

# Estrai gli archi dai path
path_edges = [[], []]  # lista di archi per ogni path
colors_path = ["green", "orange"]  # colori per i due path

for idx, path in enumerate(all_paths):

    # all_paths contiene già liste di interi
    # creo coppie di archi
    for i in range(len(path) - 1):
        path_edges[idx].append((path[i], path[i + 1]))

# disegno i nodi
nx.draw_networkx_nodes(G_sub, pos, node_size=400, node_color=node_colors)
nx.draw_networkx_labels(G_sub, pos, font_size=10, font_weight="bold")

# collego tutti i nodi
all_path_edges = set(path_edges[0] + path_edges[1])
other_edges = [e for e in G_sub.edges() if e not in all_path_edges]
nx.draw_networkx_edges(
    G_sub,
    pos,
    edgelist=other_edges,
    edge_color="gray",
    arrows=True,
    arrowsize=12,
    alpha=0.3,
)

# archi dei path
for idx, edges in enumerate(path_edges):

    nx.draw_networkx_edges(
        G_sub,
        pos,
        edgelist=edges,
        edge_color=colors_path[idx],
        arrows=True,
        arrowsize=20,
        width=2.5,
        label=f"Viaggio {idx+1}",
    )

plt.title(f"Grafo con base {active_base} - Viaggi colorati", fontsize=16)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()